# wandb-init-run composite — cx29: Trainer wires wandb.init at start + wandb.log per step

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `wandb-init-run`, `wandb-log-step`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "wandb-init-run"
DD_ATOM_IDS = ["wandb-init-run", "wandb-log-step"]
DD_SUBTOPICS = ["Logging: wandb.init run", "Logging: wandb.log step"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A wandb-instrumented Trainer is the ARENA canonical setup: open a run at the start of training, log per-batch metrics throughout. Two atoms compose:

1. **wandb-init-run** — `wandb.init(project=, name=, config=)` opens exactly ONE run per `train()` call. Goes in `pre_training_setup` (or at the very top of `fit`).
2. **wandb-log-step** — `wandb.log({'loss': ...}, step=self.step)` is called every batch from inside `training_step`. The `step=` kwarg is the global step counter (batches completed), giving wandb the x-axis.

**Anatomy.**
```python
class WandbTrainer:
    def pre_training_setup(self):
        wandb.init(                          # wandb-init-run.
            project=self.args.wandb_project,
            name=self.args.wandb_name,
            config=self.args,
        )
    def training_step(self, batch):
        ...
        wandb.log({'loss': loss.item()}, step=self.step)  # wandb-log-step.
        return loss
    def fit(self, n_epochs):
        self.pre_training_setup()
        for _ in range(n_epochs):
            for batch in self.train_loader:
                self.training_step(batch)
```

**The test mocks `wandb`.** `sys.modules['wandb']` is replaced with a `MagicMock()` so the trainer can call `wandb.init` and `wandb.log` without a real wandb install. The mock records every call so we can assert what was logged.

### Composite Exercise — Trainer wires wandb.init at start + wandb.log per step

**Atoms exercised together**: `wandb-init-run`, `wandb-log-step`

Implement `cx29_make_wandb_trainer()` which returns a `WandbTrainer` class.

Required structure:
- `WandbTrainer.__init__(self, model, optimizer, loss_fn, train_loader, args)`:
  - Store all five. `self.step = 0`. `self.history = []`.
  - `args` is a simple object with `.wandb_project`, `.wandb_name`, `.lr`, `.epochs`.
- `WandbTrainer.pre_training_setup(self)`:
  - Call `wandb.init(project=args.wandb_project, name=args.wandb_name, config=args)` (atom: wandb-init-run). EXACTLY one call.
- `WandbTrainer.training_step(self, batch)`:
  - Forward → scalar loss → zero_grad → backward → opt.step → `self.step += 1`.
  - `wandb.log({'loss': loss.item()}, step=self.step)` (atom: wandb-log-step).
  - Append `loss.item()` to `self.history`. Return scalar loss.
- `WandbTrainer.fit(self, n_epochs)`:
  - Call `pre_training_setup()` ONCE before the epoch loop.
  - For each epoch: iterate the loader, call `training_step(batch)`.

The test mocks wandb, then verifies:
- `wandb.init` called exactly once (after `fit`), with `project=`, `name=`, `config=args`.
- `wandb.log` called exactly `n_epochs * len(loader)` times.
- Each `wandb.log` call has a `step=` kwarg equal to the current step counter.

In [ ]:
def cx29_make_wandb_trainer():
    import sys
    from unittest.mock import MagicMock
    sys.modules.setdefault('wandb', MagicMock())
    import wandb

    class WandbTrainer:
        def __init__(self, model, optimizer, loss_fn, train_loader, args):
            self.model = model
            self.optimizer = optimizer
            self.loss_fn = loss_fn
            self.train_loader = train_loader
            self.args = args
            self.step = 0
            self.history = []

        def pre_training_setup(self):
            # Atom A (wandb-init-run): exactly one run per fit().
            wandb.init(
                project=self.args.wandb_project,
                name=self.args.wandb_name,
                config=self.args,
            )

        def training_step(self, batch):
            x, y = batch
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.step += 1
            # Atom B (wandb-log-step): per-batch metric with step axis.
            wandb.log({'loss': loss.item()}, step=self.step)
            self.history.append(loss.item())
            return loss

        def fit(self, n_epochs):
            self.pre_training_setup()
            for _ in range(n_epochs):
                for batch in self.train_loader:
                    self.training_step(batch)

    return WandbTrainer


<details><summary>Show solution — cx29</summary>

```python
def cx29_make_wandb_trainer():
    import sys
    from unittest.mock import MagicMock
    sys.modules.setdefault('wandb', MagicMock())
    import wandb

    class WandbTrainer:
        def __init__(self, model, optimizer, loss_fn, train_loader, args):
            self.model = model
            self.optimizer = optimizer
            self.loss_fn = loss_fn
            self.train_loader = train_loader
            self.args = args
            self.step = 0
            self.history = []

        def pre_training_setup(self):
            # Atom A (wandb-init-run): exactly one run per fit().
            wandb.init(
                project=self.args.wandb_project,
                name=self.args.wandb_name,
                config=self.args,
            )

        def training_step(self, batch):
            x, y = batch
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.step += 1
            # Atom B (wandb-log-step): per-batch metric with step axis.
            wandb.log({'loss': loss.item()}, step=self.step)
            self.history.append(loss.item())
            return loss

        def fit(self, n_epochs):
            self.pre_training_setup()
            for _ in range(n_epochs):
                for batch in self.train_loader:
                    self.training_step(batch)

    return WandbTrainer
```

Order matters: `wandb.init` must fire BEFORE the first `wandb.log` or wandb errors out with 'no run in progress'. Putting `init` in `pre_training_setup` (called once at the top of `fit`) guarantees the ordering. The `step=` kwarg pins wandb's x-axis to your training step — without it, wandb infers a step automatically and can lose alignment across runs.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx29'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx29',
        'subtopics': ["Logging: wandb.init run", "Logging: wandb.log step"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()